In [17]:
import polars as pl
import unicodedata
import re
from datetime import datetime

In [122]:
# Required Column of Catalog
REQUIRED_CATALOG_COLUMNS = [
    'GROUPED DIVISION', 'DIVISION', 'DEPARTMENT', 'GOLD CODE', 'TILLCODE', 'SV',
    'PRODUCT NAME VIE', 'UNIT', 'START DATE', 'END DATE', 'SYSTEM GOLD VLL', 'REGION NORTH SOUTH CENTER ALL',
    'PAGE', 'PAGE DIGITAL', 'THEMATIC', 'POSITION', 'NORMAL SELLING PRICE VND', 'PROMOTIONAL SELLING PRICE VND',
    'MULTIBUYCOMBO SELLING PRICE VND', 'DISCOUNT TO CUSTOMER VND', '% DISCOUNT TO CUSTOMER', 'PROMO TYPE',
    'PROMO DETAILS', 'PROMOTION GIFT', 'GIFT VALUE', 'NOTE', 'THE1 POINTS', 'THE1 EXCLUSIVE PRICE VND', 'NOTE_1']

# Required Header Columns
HEADER_COLUMNS = [
    'GROUPED DIVISION', 'DIVISION', 'DEPARTMENT', 'GOLD CODE', 'TILLCODE', 'SV'
]

# Required Data Columns
REQUIRED_DATA_COLUMNS = [
    'GOLD CODE', 'SV', 'NORMAL SELLING PRICE VND', 'PROMOTIONAL SELLING PRICE VND'
]

In [112]:
# Normalize Function
def normalize(value):
    if value is None:
        return None

    value = unicodedata.normalize("NFKC", str(value))
    value = re.sub(r"[/\r\n\t\f\v]+", " ", value)
    value = re.sub(r"[^\w\s%]", "", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip().upper()

# Mapping Column
def map_columns(df, required_columns):
    mapping = {}
    used = set()

    for required in required_columns:
        if required in df.columns:
            used.add(required)
            continue

        matches = [
            col for col in df.columns
            if col.startswith(required + " ") and col not in used
        ]

        if len(matches) == 1:
            mapping[matches[0]] = required
            used.add(matches[0])

    df = df.rename(mapping)

    return df.select([
        pl.col(col) if col in df.columns
        else pl.lit(None, dtype=pl.String).alias(col)
        for col in required_columns
    ])

# Scan Header
def scan_header(pth, sheet_name):
    data = pl.read_excel(
        pth,
        infer_schema_length=0,
        sheet_name=sheet_name
    )

    has_header = False

    for i in range(0, 10, 1):
        if all(col in data.row(i) for col in HEADER_COLUMNS):
            has_header = True
            break

    if has_header:
        return i + 1
    else:
        raise ValueError(f"Could not find header of sheet {sheet_name} !!!")

In [113]:
file_path = "/content/C621.xlsx"
sheet_name = "MB+VINH"
target_date = datetime.strptime("08/10/2026", "%d/%m/%Y").date()

In [145]:
df = pl.read_excel(
    file_path,
    infer_schema_length=0,
    read_options={"header_row": scan_header(file_path, sheet_name)},
    sheet_name=sheet_name
)

In [146]:
df = df.rename({col: normalize(col) for col in df.columns})
df = map_columns(df, REQUIRED_CATALOG_COLUMNS)

In [147]:
df = df.drop_nulls(subset=REQUIRED_DATA_COLUMNS)

In [148]:
df = df.with_columns(
    pl.lit(sheet_name).alias("REGION")
)

In [152]:
df = df.with_columns(
    pl.coalesce(
        pl.col("PROMOTIONAL SELLING PRICE VND"),
        pl.col("NORMAL SELLING PRICE VND")
    )
    .fill_null("None Price")
    .alias("FINAL PRICE")
)

In [154]:
df = df.with_columns(
    pl.lit("112").alias("SUGGEST SITE")
)

In [157]:
df = df.with_columns(
    pl.concat_str(
        ["GOLD CODE", "SV", "SUGGEST SITE"],
        separator="-"
    ).alias("KEY")
)

In [163]:
df = df.filter(
    pl.col('FINAL PRICE') != 'None Price'
)

In [164]:
df = df.with_columns(pl.col('FINAL PRICE').cast(pl.Int64))

In [195]:
query = pl.read_excel('/content/query_price.xlsx', infer_schema_length=0)

In [196]:
query = query.with_columns(
    pl.concat_str(
        ["GOLD_CODE", "SV", "CH"],
        separator="-"
    ).alias("KEY")
)

In [197]:
query = query.with_columns(
    pl.col("START_DATE").str.strptime(pl.Date, format="%d/%m/%y", strict=False).alias("_NORMAL_START_DATE"),
    pl.col("END_DATE").str.strptime(pl.Date, format="%d/%m/%y", strict=False).alias("_NORMAL_END_DATE"),
    pl.col("START_DATE_KM").str.strptime(pl.Date, format="%d/%m/%y", strict=False).alias("_PROMO_START_DATE"),
    pl.col("END_DATE_KM").str.strptime(pl.Date, format="%d/%m/%y", strict=False).alias("_PROMO_END_DATE"),
)

normal_is_valid = (
    pl.col("GIA_NORMAL").is_not_null()
    & (pl.col("_NORMAL_START_DATE") <= pl.lit(target_date))
    & (pl.col("_NORMAL_END_DATE") >= pl.lit(target_date))
)
promo_is_valid = (
    pl.col("GIA_KM").is_not_null()
    & (pl.col("_PROMO_START_DATE") <= pl.lit(target_date))
    & (pl.col("_PROMO_END_DATE") >= pl.lit(target_date))
)

query = query.with_columns(
    pl.when(promo_is_valid).then(pl.col("GIA_KM")).when(normal_is_valid).then(pl.col("GIA_NORMAL")).alias("FINAL PRICE"),
    pl.when(promo_is_valid).then(pl.col("_PROMO_START_DATE")).when(normal_is_valid).then(pl.col("_NORMAL_START_DATE")).alias("VALID START DATE"),
    pl.when(promo_is_valid).then(pl.col("_PROMO_END_DATE")).when(normal_is_valid).then(pl.col("_NORMAL_END_DATE")).alias("VALID END DATE"),
    pl.when(promo_is_valid).then(pl.lit("KM")).when(normal_is_valid).then(pl.lit("NORMAL")).alias("PRICE TYPE"),
).drop("_NORMAL_START_DATE", "_NORMAL_END_DATE", "_PROMO_START_DATE", "_PROMO_END_DATE")

In [198]:
query = query.filter(
    (pl.col("DIVISION") != "DIVISION") &
    (pl.col("FINAL PRICE").is_not_null()) &
    (pl.col("VALID START DATE").is_not_null()) &
    (pl.col("VALID END DATE").is_not_null())
)

In [199]:
query = query.with_columns(pl.col('FINAL PRICE').cast(pl.Int64))

In [201]:
query = query.with_columns(
    pl.col(["VALID START DATE", "VALID END DATE"])
    .dt.strftime("%d/%m/%Y")
)

In [202]:
query

DIVISION,DEPT,FAMILY_DESC,CH,GOLD_CODE,SV,DESCRIPTION,PRICELIST_NORMAL,GIA_NORMAL,START_DATE,END_DATE,PRICELIST_KM,GIA_KM,START_DATE_KM,END_DATE_KM,ATT_CLASS,ATT_CODE,VAT,KEY,FINAL PRICE,VALID START DATE,VALID END DATE
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str
"""01""","""0110""","""01101011-BEAUTY CARE""","""101""","""00445674""","""1""","""SAY TOC PHILIPS HP8108/00""","""10900101""","""378000""","""23/06/26""","""31/12/49""","""20300101""","""269000""","""07/10/26""","""21/10/26""",null,null,"""5""","""00445674-1-101""",269000,"""07/10/2026""","""21/10/2026"""
"""01""","""0110""","""01101011-BEAUTY CARE""","""102""","""00445674""","""1""","""SAY TOC PHILIPS HP8108/00""","""10900102""","""378000""","""23/06/26""","""31/12/49""","""20300102""","""269000""","""07/10/26""","""21/10/26""",null,null,"""5""","""00445674-1-102""",269000,"""07/10/2026""","""21/10/2026"""
"""01""","""0110""","""01101011-BEAUTY CARE""","""103""","""00445674""","""1""","""SAY TOC PHILIPS HP8108/00""","""10900103""","""378000""","""23/06/26""","""31/12/49""","""20300103""","""269000""","""07/10/26""","""21/10/26""",null,null,"""5""","""00445674-1-103""",269000,"""07/10/2026""","""21/10/2026"""
"""01""","""0110""","""01101011-BEAUTY CARE""","""104""","""00445674""","""1""","""SAY TOC PHILIPS HP8108/00""","""10900104""","""378000""","""23/06/26""","""31/12/49""","""20300104""","""269000""","""07/10/26""","""21/10/26""",null,null,"""5""","""00445674-1-104""",269000,"""07/10/2026""","""21/10/2026"""
"""01""","""0110""","""01101011-BEAUTY CARE""","""106""","""00445674""","""1""","""SAY TOC PHILIPS HP8108/00""","""10900106""","""378000""","""23/06/26""","""31/12/49""","""20300106""","""269000""","""07/10/26""","""21/10/26""",null,null,"""5""","""00445674-1-106""",269000,"""07/10/2026""","""21/10/2026"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""04""","""0450""","""04505002-ADULT FACIAL TISSUE""","""154""","""02106988""","""1""","""LO 6 KH.GIAY UOT MINI TEMPO PR…","""10900154""","""56000""","""10/06/26""","""31/12/49""","""20300154""","""34900""","""08/10/26""","""21/10/26""",null,null,"""5""","""02106988-1-154""",34900,"""08/10/2026""","""21/10/2026"""
"""04""","""0450""","""04505002-ADULT FACIAL TISSUE""","""155""","""02106988""","""1""","""LO 6 KH.GIAY UOT MINI TEMPO PR…","""10900155""","""56000""","""10/06/26""","""31/12/49""","""20300155""","""34900""","""08/10/26""","""21/10/26""",null,null,"""5""","""02106988-1-155""",34900,"""08/10/2026""","""21/10/2026"""
"""04""","""0450""","""04505002-ADULT FACIAL TISSUE""","""156""","""02106988""","""1""","""LO 6 KH.GIAY UOT MINI TEMPO PR…","""10900156""","""56000""","""10/06/26""","""31/12/49""",null,null,null,null,null,null,"""5""","""02106988-1-156""",56000,"""10/06/2026""","""31/12/2049"""
